In [ ]:
import pandas as pd
import numpy as np
import requests
import zipfile
import os
import re
import sys
import contextlib
from datetime import datetime, timedelta
from tqdm import tqdm
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, Alignment, PatternFill

# ==============================================================================
# ⚙️ 1. CONFIGURAÇÕES
# ==============================================================================

ANO_INICIO = 2021
ANO_FIM = 2025

BASE_DIR = os.path.join(os.getcwd(), "pipeline_b3_cvm")
RAW_DIR = os.path.join(BASE_DIR, "temp_cache")  # Pasta temporária
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

CAMINHO_TICKERS = r"BASE_EMPRESAS_TICKERS.csv"
CVM_BASE_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/"
# Link direto B3 (sujeito a alterações pela B3, fallback manual implementado)
B3_URL_TEMPLATE = "https://bvmf.bmfbovespa.com.br/InstDados/SerHist/COTAHIST_A{}.ZIP"

# ==============================================================================
# 🛠️ 2. FUNÇÕES UTILITÁRIAS
# ==============================================================================

def setup_directories():
    for path in [RAW_DIR, OUTPUT_DIR]:
        os.makedirs(path, exist_ok=True)
    print(f"📂 Diretórios configurados em: {BASE_DIR}")

def limpar_cnpj(cnpj):
    return re.sub(r"\D", "", str(cnpj))

def converter_escala(valor, escala):
    if pd.isna(valor):
        return 0.0
    if escala == "MIL":
        return valor * 1000.0
    return valor

def limpar_cache(ano):
    """Remove arquivos temporários para economizar disco"""
    for f in os.listdir(RAW_DIR):
        if str(ano) in f and (f.endswith(".zip") or f.endswith(".ZIP")):
            try:
                os.remove(os.path.join(RAW_DIR, f))
            except:
                pass

def carregar_tickers(caminho):
    print(f"\n📖 Carregando Mapeamento de Tickers...")
    if not os.path.exists(caminho):
        print("❌ ERRO: Arquivo de tickers não encontrado.")
        return {}, [], {}

    encodings = ["utf-8-sig", "latin1", "cp1252"]
    for enc in encodings:
        try:
            df = pd.read_csv(caminho, sep=";", dtype=str, encoding=enc)
            df.columns = df.columns.str.upper().str.strip()

            # ---- Mínima mudança: capturar TIPO (setor) se existir
            has_tipo = "TIPO" in df.columns

            if "CNPJ" in df.columns and "TICKER" in df.columns:
                df["CNPJ_CLEAN"] = df["CNPJ"].apply(limpar_cnpj)
                df["TICKER"] = df["TICKER"].str.strip().str.upper()

                if has_tipo:
                    df["TIPO"] = df["TIPO"].astype(str).str.strip()

                mapa_ticker = dict(zip(df["CNPJ_CLEAN"], df["TICKER"]))
                lista_tickers = df["TICKER"].unique().tolist()

                mapa_tipo = {}
                if has_tipo:
                    # Se tiver múltiplos tipos por CNPJ, pega o primeiro não-nulo (pragmático)
                    tmp = (
                        df[["CNPJ_CLEAN", "TIPO"]]
                        .dropna(subset=["CNPJ_CLEAN"])
                        .drop_duplicates(subset=["CNPJ_CLEAN"])
                    )
                    mapa_tipo = dict(zip(tmp["CNPJ_CLEAN"], tmp["TIPO"]))

                return mapa_ticker, lista_tickers, mapa_tipo
        except:
            continue

    return {}, [], {}

# ==============================================================================
# 📊 3. ENGINE B3 (DATA LAKE DE PREÇOS)
# ==============================================================================

def download_b3_cotahist(ano):
    filename = f"COTAHIST_A{ano}.ZIP"
    path = os.path.join(RAW_DIR, filename)

    # Se já existe (colocado manualmente ou download anterior), usa ele
    if os.path.exists(path):
        return path

    print(f"   ⬇️  Baixando Cotações B3 {ano} (Pode demorar um pouco)...")
    url = B3_URL_TEMPLATE.format(ano)

    try:
        # Tenta download oficial (verificando SSL false pois certificado B3 as vezes falha em scripts)
        r = requests.get(url, stream=True, verify=False)
        if r.status_code == 200:
            with open(path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024):
                    f.write(chunk)
            return path
        else:
            print(f"   ⚠️  B3 retornou erro {r.status_code}. Tente baixar manualmente.")
            return None
    except Exception as e:
        print(f"   ⚠️  Falha no download B3: {e}")
        return None

def processar_precos_b3(anos, tickers_interesse):
    print(f"\n💹 CONSTRUINDO BASE DE PREÇOS (B3)...")
    dfs_precos = []

    # Set para busca rápida O(1)
    set_tickers = set(tickers_interesse)

    for ano in anos:
        path_zip = download_b3_cotahist(ano)
        if not path_zip:
            print(f"   ❌ Pulei ano {ano} (Arquivo B3 não encontrado).")
            continue

        try:
            with zipfile.ZipFile(path_zip, "r") as z:
                # O arquivo dentro geralmente chama COTAHIST_A{ANO}.TXT
                arquivo_txt = next((f for f in z.namelist() if f.endswith(".TXT")), None)
                if not arquivo_txt:
                    continue

                print(f"   ⚙️  Lendo {arquivo_txt} em memória...")

                with z.open(arquivo_txt) as f:
                    # Layout COTAHIST (Posicional)
                    # Data: 02-10 (YYYYMMDD)
                    # CodNeg (Ticker): 12-24
                    # PrecoUlt (Fechamento): 108-121 (2 decimais implicitos)

                    colspecs = [(2, 10), (12, 24), (108, 121)]
                    names = ["DATA", "TICKER", "PRECO"]

                    reader = pd.read_fwf(
                        f,
                        colspecs=colspecs,
                        names=names,
                        header=None,
                        chunksize=100000,
                        encoding="latin1",
                    )

                    chunks = []
                    for chunk in reader:
                        chunk["TICKER"] = chunk["TICKER"].str.strip()
                        chunk = chunk[chunk["TICKER"].isin(set_tickers)].copy()

                        if not chunk.empty:
                            chunk["DATA"] = pd.to_datetime(chunk["DATA"], format="%Y%m%d", errors="coerce")
                            chunk["PRECO"] = chunk["PRECO"] / 100.0
                            chunks.append(chunk)

                    if chunks:
                        dfs_precos.append(pd.concat(chunks))

        except Exception as e:
            print(f"   ❌ Erro ao processar B3 {ano}: {e}")

    if not dfs_precos:
        return pd.DataFrame()

    df_final = pd.concat(dfs_precos).sort_values("DATA")
    print(f"   ✅ Base de Preços carregada: {len(df_final)} registros.")
    return df_final

# ==============================================================================
# 🧮 4. ENGINE CVM (ZERO FOOTPRINT)
# ==============================================================================

def ler_csv_do_zip(path_zip, nome_parcial, colunas):
    if not path_zip or not os.path.exists(path_zip):
        return pd.DataFrame()
    try:
        with zipfile.ZipFile(path_zip, "r") as z:
            arquivo = next(
                (f for f in z.namelist() if nome_parcial.lower() in f.lower() and f.endswith(".csv")),
                None,
            )
            if not arquivo:
                return pd.DataFrame()

            with z.open(arquivo) as f:
                cabecalho = pd.read_csv(f, sep=";", encoding="latin1", nrows=0)
                cols_ok = [c for c in colunas if c in cabecalho.columns]

            with z.open(arquivo) as f:
                df = pd.read_csv(f, sep=";", encoding="latin1", usecols=cols_ok, low_memory=False)

        if "ORDEM_EXERC" in df.columns:
            df = df[df["ORDEM_EXERC"] == "ÚLTIMO"]
        if "DT_FIM_EXERC" in df.columns:
            df["DT_FIM_EXERC"] = pd.to_datetime(df["DT_FIM_EXERC"])
        if "DT_INI_EXERC" in df.columns:
            df["DT_INI_EXERC"] = pd.to_datetime(df["DT_INI_EXERC"])
        if "VL_CONTA" in df.columns:
            df["VL_REAL"] = df.apply(lambda x: converter_escala(x["VL_CONTA"], x.get("ESCALA_MOEDA")), axis=1)
        if "CNPJ_CIA" in df.columns:
            df["CNPJ_CLEAN"] = df["CNPJ_CIA"].apply(limpar_cnpj)

        return df
    except:
        return pd.DataFrame()

def download_zip_cvm(tipo, ano):
    filename = f"{tipo.lower()}_cia_aberta_{ano}.zip"
    url = f"{CVM_BASE_URL}{tipo}/DADOS/{filename}"
    path = os.path.join(RAW_DIR, filename)
    if os.path.exists(path):
        return path

    try:
        r = requests.get(url, stream=True)
        if r.status_code == 200:
            with open(path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024):
                    f.write(chunk)
            return path
    except:
        pass
    return None

# --- Helpers de Extração ---
def extrair_patrimonio(bpa, bpp, data):
    bpa_d = bpa[bpa["DT_FIM_EXERC"] == data]
    bpp_d = bpp[bpp["DT_FIM_EXERC"] == data]
    if bpa_d.empty:
        return pd.DataFrame()

    res = pd.DataFrame()
    res["CNPJ_CLEAN"] = bpa_d["CNPJ_CLEAN"].unique()

    def get_valor(df_fonte, conta):
        temp = df_fonte[df_fonte["CD_CONTA"] == conta].groupby("CNPJ_CLEAN")["VL_REAL"].sum().reset_index()
        return temp.set_index("CNPJ_CLEAN")["VL_REAL"]

    def get_grupo(df_fonte, lista_contas):
        temp = df_fonte[df_fonte["CD_CONTA"].isin(lista_contas)].groupby("CNPJ_CLEAN")["VL_REAL"].sum().reset_index()
        return temp.set_index("CNPJ_CLEAN")["VL_REAL"]

    res = res.set_index("CNPJ_CLEAN")
    res["Ativo Total"] = get_valor(bpa_d, "1")
    res["Caixa"] = get_valor(bpa_d, "1.01.01")
    res["Patrimonio Liquido"] = get_valor(bpp_d, "2.03")
    res["Divida Bruta"] = get_grupo(bpp_d, ["2.01.04", "2.02.01"])

    res = res.reset_index()
    res["Divida Liquida"] = res["Divida Bruta"].fillna(0) - res["Caixa"].fillna(0)
    return res

def extrair_dre_periodo(dre, data, is_q4_calc=False):
    if not is_q4_calc:
        d = dre[dre["DT_FIM_EXERC"] == data]
    else:
        d = dre

    if d.empty:
        return pd.DataFrame()

    def get_valor(conta_base):
        mask = d["CD_CONTA"].astype(str).str.startswith(conta_base)

        if conta_base == "3.99":  # EPS
            mask = d["CD_CONTA"] == "3.99.01.01"
            if d[mask].empty:
                mask = d["CD_CONTA"] == "3.99"
        elif conta_base in ["3.09", "3.11"]:
            mask = d["CD_CONTA"].isin(["3.09", "3.11", "3.07"])

        return d[mask].groupby("CNPJ_CLEAN")["VL_REAL"].sum()

    res = pd.DataFrame()
    res["CNPJ_CLEAN"] = d["CNPJ_CLEAN"].unique()
    res = res.set_index("CNPJ_CLEAN")

    res["Lucro Liquido"] = get_valor("3.09")
    res["Res_Fin"] = d[d["CD_CONTA"] == "3.06"].groupby("CNPJ_CLEAN")["VL_REAL"].sum()
    res["IR"] = d[d["CD_CONTA"] == "3.08"].groupby("CNPJ_CLEAN")["VL_REAL"].sum()
    res["LPA"] = get_valor("3.99")

    return res.reset_index()

def extrair_dfc_periodo(dfc, data, is_q4_calc=False):
    if not is_q4_calc:
        d = dfc[dfc["DT_FIM_EXERC"] == data]
    else:
        d = dfc

    if d.empty:
        return pd.DataFrame(columns=["CNPJ_CLEAN", "DA_Trimestral"])

    mask = d["CD_CONTA"].str.startswith("6.01.01")
    if "DS_CONTA" in d.columns:
        ds = d["DS_CONTA"].astype(str).str.lower()
        mask = mask & (ds.str.contains("deprecia") | ds.str.contains("amortiza"))

    da = d[mask].groupby("CNPJ_CLEAN")["VL_REAL"].sum()
    return pd.DataFrame({"DA_Trimestral": da}).reset_index()

def processar_ano_cvm(ano):
    print(f"\n⚙️  Processando CVM {ano}...")

    path_itr = download_zip_cvm("ITR", ano)
    path_dfp = download_zip_cvm("DFP", ano)

    cols = [
        "CNPJ_CIA",
        "DENOM_CIA",
        "DT_FIM_EXERC",
        "DT_INI_EXERC",
        "CD_CONTA",
        "DS_CONTA",
        "VL_CONTA",
        "ESCALA_MOEDA",
        "ORDEM_EXERC",
    ]

    itr_dre = ler_csv_do_zip(path_itr, "DRE_con", cols)
    itr_dfc = ler_csv_do_zip(path_itr, "DFC_MI_con", cols)
    itr_bpa = ler_csv_do_zip(path_itr, "BPA_con", cols)
    itr_bpp = ler_csv_do_zip(path_itr, "BPP_con", cols)

    if itr_dre.empty:
        return pd.DataFrame()

    itr_dre["DIAS"] = (itr_dre["DT_FIM_EXERC"] - itr_dre["DT_INI_EXERC"]).dt.days
    mask_trim = (itr_dre["DIAS"] >= 80) & (itr_dre["DIAS"] <= 100)
    dre_q123 = itr_dre[mask_trim].copy()

    resultados = []

    datas = sorted(pd.to_datetime(itr_bpa["DT_FIM_EXERC"].unique()))
    for data in datas:
        if data.month == 12:
            continue

        pat = extrair_patrimonio(itr_bpa, itr_bpp, data)
        if pat.empty:
            continue

        dre = extrair_dre_periodo(dre_q123, data)
        dfc = extrair_dfc_periodo(itr_dfc, data)

        res = pat.merge(dre, on="CNPJ_CLEAN", how="left").merge(dfc, on="CNPJ_CLEAN", how="left")
        res["DT_FIM_EXERC"] = data
        resultados.append(res)

    # Q4 via DFP
    dfp_dre = ler_csv_do_zip(path_dfp, "DRE_con", cols)
    dfp_dfc = ler_csv_do_zip(path_dfp, "DFC_MI_con", cols)
    dfp_bpa = ler_csv_do_zip(path_dfp, "BPA_con", cols)
    dfp_bpp = ler_csv_do_zip(path_dfp, "BPP_con", cols)

    if not dfp_dre.empty:
        mask_9m = (itr_dre["DT_INI_EXERC"].dt.month == 1) & (itr_dre["DT_FIM_EXERC"].dt.month == 9)
        dre_9m = itr_dre[mask_9m][["CNPJ_CLEAN", "CD_CONTA", "VL_REAL"]].rename(columns={"VL_REAL": "VL_9M"})

        dfp_12m = dfp_dre[dfp_dre["DT_INI_EXERC"].dt.month == 1][["CNPJ_CLEAN", "CD_CONTA", "VL_REAL"]]

        calc = dfp_12m.merge(dre_9m, on=["CNPJ_CLEAN", "CD_CONTA"], how="left")
        calc["VL_9M"] = calc["VL_9M"].fillna(0)
        calc["VL_REAL"] = calc["VL_REAL"] - calc["VL_9M"]

        dre_q4 = extrair_dre_periodo(calc, None, is_q4_calc=True)
        pat_q4 = extrair_patrimonio(dfp_bpa, dfp_bpp, pd.Timestamp(f"{ano}-12-31"))

        dfc_q4 = pd.DataFrame(columns=["CNPJ_CLEAN", "DA_Trimestral"])

        if not pat_q4.empty:
            res_q4 = pat_q4.merge(dre_q4, on="CNPJ_CLEAN", how="left").merge(dfc_q4, on="CNPJ_CLEAN", how="left")
            res_q4["DT_FIM_EXERC"] = pd.Timestamp(f"{ano}-12-31")
            resultados.append(res_q4)

    limpar_cache(ano)

    if not resultados:
        return pd.DataFrame()

    final = pd.concat(resultados, ignore_index=True)

    nomes = itr_bpa[["CNPJ_CLEAN", "DENOM_CIA"]].drop_duplicates("CNPJ_CLEAN")
    return final.merge(nomes, on="CNPJ_CLEAN", how="left")

# ==============================================================================
# 🔗 5. JOIN FINAL (CVM + B3)
# ==============================================================================

def consolidar_dados(df_cvm, df_precos, mapa_tickers, mapa_tipo):
    print(f"\n🔗 CRUZANDO DADOS (Merge AsOf)...")

    df_cvm["Ticker"] = df_cvm["CNPJ_CLEAN"].map(mapa_tickers)

    # ---- Mínima mudança: trazer Setor (TIPO) para o dataframe
    df_cvm["Setor"] = df_cvm["CNPJ_CLEAN"].map(mapa_tipo)

    df_cvm = df_cvm[df_cvm["Ticker"].notna()].copy()

    df_cvm = df_cvm.sort_values("DT_FIM_EXERC")

    if df_precos.empty:
        df_final = df_cvm.copy()
        df_final["DATA"] = pd.NaT
        df_final["PRECO"] = np.nan
    else:
        df_precos = df_precos.sort_values("DATA")

        df_final = pd.merge_asof(
            df_cvm,
            df_precos,
            left_on="DT_FIM_EXERC",
            right_on="DATA",
            by="Ticker",
            direction="backward",
            tolerance=pd.Timedelta(days=10),
        )

    # EBITDA
    df_final["EBITDA"] = (
        df_final["Lucro Liquido"].fillna(0)
        - df_final["Res_Fin"].fillna(0)
        - df_final["IR"].fillna(0)
        + df_final["DA_Trimestral"].fillna(0)
    )

    # Ações Implícitas
    df_final["Acoes_Implicitas"] = np.where(
        (df_final["LPA"].abs() > 0.01),
        df_final["Lucro Liquido"] / df_final["LPA"],
        np.nan,
    )

    # Market Cap
    df_final["Market_Cap_Calc"] = df_final["PRECO"] * df_final["Acoes_Implicitas"]
    df_final["Price_to_Book"] = df_final["Market_Cap_Calc"] / df_final["Patrimonio Liquido"]

    # EV e múltiplos
    df_final["EV"] = df_final["Market_Cap_Calc"].fillna(0) + df_final["Divida Liquida"].fillna(0)
    df_final["P_E"] = df_final["Market_Cap_Calc"] / (df_final["Lucro Liquido"] * 4)
    df_final["EV_EBITDA"] = df_final["EV"] / (df_final["EBITDA"] * 4)

    return df_final

# ==============================================================================
# 📌 6. RESUMOS (SETORES + MERCADO)
# ==============================================================================

def _safe_median(s):
    s = pd.to_numeric(s, errors="coerce")
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if s.empty:
        return np.nan
    return s.median()

def _safe_sum(s):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return np.nan_to_num(s, nan=0.0).sum()

def _weighted_pe(market_cap, lucro):
    # Proxy: P/E agregado = sum(MktCap) / (4*sum(Lucro)) com lucro>0
    mc = pd.to_numeric(market_cap, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    ll = pd.to_numeric(lucro, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    ll_pos = ll.where(ll > 0, 0.0)
    denom = 4.0 * ll_pos.sum()
    if denom <= 0:
        return np.nan
    return mc.sum() / denom

def _weighted_ev_ebitda(ev, ebitda):
    # Proxy: EV/EBITDA agregado = sum(EV) / (4*sum(EBITDA)) com EBITDA>0
    evv = pd.to_numeric(ev, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    eb = pd.to_numeric(ebitda, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    eb_pos = eb.where(eb > 0, 0.0)
    denom = 4.0 * eb_pos.sum()
    if denom <= 0:
        return np.nan
    return evv.sum() / denom

def gerar_resumos_snapshot(df_final):
    """
    Gera:
      - resumo_setores: 1 linha por setor, usando o último DT_FIM_EXERC (snapshot)
      - resumo_mercado: 1 linha agregada, mesmo snapshot
    """
    if df_final.empty or "DT_FIM_EXERC" not in df_final.columns:
        return pd.DataFrame(), pd.DataFrame()

    snapshot_date = df_final["DT_FIM_EXERC"].max()
    snap = df_final[df_final["DT_FIM_EXERC"] == snapshot_date].copy()

    # Se não tiver Setor, ainda gera Mercado
    if "Setor" not in snap.columns:
        snap["Setor"] = np.nan

    # Limpa setor vazio
    snap["Setor"] = snap["Setor"].replace("", np.nan)

    # --- Resumo Setores (1 linha por setor)
    setores = (
        snap.dropna(subset=["Setor"])
        .groupby("Setor", dropna=True)
        .apply(lambda g: pd.Series({
            "DT_FIM_EXERC": snapshot_date,
            "N_Empresas": g["Ticker"].nunique() if "Ticker" in g.columns else len(g),
            "Market_Cap_Sum": _safe_sum(g["Market_Cap_Calc"]) if "Market_Cap_Calc" in g.columns else np.nan,
            "EV_Sum": _safe_sum(g["EV"]) if "EV" in g.columns else np.nan,
            "Lucro_Sum": _safe_sum(g["Lucro Liquido"]) if "Lucro Liquido" in g.columns else np.nan,
            "EBITDA_Sum": _safe_sum(g["EBITDA"]) if "EBITDA" in g.columns else np.nan,
            "Price_to_Book_Median": _safe_median(g["Price_to_Book"]) if "Price_to_Book" in g.columns else np.nan,
            "P_E_Median": _safe_median(g["P_E"]) if "P_E" in g.columns else np.nan,
            "EV_EBITDA_Median": _safe_median(g["EV_EBITDA"]) if "EV_EBITDA" in g.columns else np.nan,
            "P_E_Agregado": _weighted_pe(g.get("Market_Cap_Calc"), g.get("Lucro Liquido")),
            "EV_EBITDA_Agregado": _weighted_ev_ebitda(g.get("EV"), g.get("EBITDA")),
        }))
        .reset_index()
        .rename(columns={"Setor": "SETOR"})
    )

    # --- Resumo Mercado (1 linha)
    mercado = pd.DataFrame([{
        "DT_FIM_EXERC": snapshot_date,
        "N_Empresas": snap["Ticker"].nunique() if "Ticker" in snap.columns else len(snap),
        "Market_Cap_Sum": _safe_sum(snap["Market_Cap_Calc"]) if "Market_Cap_Calc" in snap.columns else np.nan,
        "EV_Sum": _safe_sum(snap["EV"]) if "EV" in snap.columns else np.nan,
        "Lucro_Sum": _safe_sum(snap["Lucro Liquido"]) if "Lucro Liquido" in snap.columns else np.nan,
        "EBITDA_Sum": _safe_sum(snap["EBITDA"]) if "EBITDA" in snap.columns else np.nan,
        "Price_to_Book_Median": _safe_median(snap["Price_to_Book"]) if "Price_to_Book" in snap.columns else np.nan,
        "P_E_Median": _safe_median(snap["P_E"]) if "P_E" in snap.columns else np.nan,
        "EV_EBITDA_Median": _safe_median(snap["EV_EBITDA"]) if "EV_EBITDA" in snap.columns else np.nan,
        "P_E_Agregado": _weighted_pe(snap.get("Market_Cap_Calc"), snap.get("Lucro Liquido")),
        "EV_EBITDA_Agregado": _weighted_ev_ebitda(snap.get("EV"), snap.get("EBITDA")),
    }])

    return setores, mercado

# ==============================================================================
# 💾 7. EXPORT
# ==============================================================================

def exportar_excel(df_final):
    arquivo = os.path.join(OUTPUT_DIR, f"Valuation_ZeroYahoo_{datetime.now().strftime('%Y%m%d')}.xlsx")
    print(f"\n💾 Salvando: {arquivo}")

    wb = Workbook()

    # -----------------
    # Aba 1: Dados
    # -----------------
    ws = wb.active
    ws.title = "Dados"

    cols = [
        "DENOM_CIA", "Ticker", "Setor", "DT_FIM_EXERC", "DATA_PRECO",
        "Ativo Total", "Caixa", "Divida Bruta", "Divida Liquida", "Patrimonio Liquido",
        "Lucro Liquido", "LPA", "EBITDA", "PRECO", "Acoes_Implicitas", "Market_Cap_Calc",
        "EV", "P_E", "EV_EBITDA", "Res_Fin", "IR", "DA_Trimestral", "Price_to_Book"
    ]

    df_out = df_final.rename(columns={"DATA": "DATA_PRECO"}).copy()
    final_cols = [c for c in cols if c in df_out.columns]

    for r in dataframe_to_rows(df_out[final_cols], index=False, header=True):
        ws.append(r)

    header_fill = PatternFill(start_color="366092", end_color="366092", fill_type="solid")
    header_font = Font(color="FFFFFF", bold=True)
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font

    # -----------------
    # Aba 2: Resumo Setores (snapshot)
    # -----------------
    resumo_setores, resumo_mercado = gerar_resumos_snapshot(df_out)

    ws2 = wb.create_sheet("Resumo_Setores")
    if not resumo_setores.empty:
        for r in dataframe_to_rows(resumo_setores, index=False, header=True):
            ws2.append(r)
        for cell in ws2[1]:
            cell.fill = header_fill
            cell.font = header_font
    else:
        ws2.append(["Sem dados de setor (coluna 'tipo'/'TIPO' ausente ou sem match)."])

    # -----------------
    # Aba 3: Resumo Mercado (snapshot)
    # -----------------
    ws3 = wb.create_sheet("Resumo_Mercado")
    if not resumo_mercado.empty:
        for r in dataframe_to_rows(resumo_mercado, index=False, header=True):
            ws3.append(r)
        for cell in ws3[1]:
            cell.fill = header_fill
            cell.font = header_font
    else:
        ws3.append(["Sem dados para resumo de mercado."])

    wb.save(arquivo)
    print("✅ Processo Finalizado com Sucesso!")

# ==============================================================================
# MAIN
# ==============================================================================

if __name__ == "__main__":
    requests.packages.urllib3.disable_warnings()

    setup_directories()

    # 1. Carrega Tickers + Setor (TIPO)
    mapa_tickers, lista_tickers, mapa_tipo = carregar_tickers(CAMINHO_TICKERS)

    if not mapa_tickers:
        print("❌ Abortando.")
        exit()

    anos = range(ANO_INICIO, ANO_FIM + 1)

    # 2. Monta Data Lake de Preços (B3)
    df_precos = processar_precos_b3(anos, lista_tickers)

    if df_precos.empty:
        print("⚠️  ATENÇÃO: Nenhuma cotação foi carregada da B3.")
        print("    Verifique se os arquivos COTAHIST_A{ANO}.ZIP estão na pasta ou se o download funcionou.")
        print("    Sem preços, o relatório virá apenas com dados contábeis.")

    # 3. Processa CVM (Contábil)
    dfs_cvm = []
    for ano in anos:
        df = processar_ano_cvm(ano)
        if not df.empty:
            dfs_cvm.append(df)

    if dfs_cvm:
        df_cvm_full = pd.concat(dfs_cvm, ignore_index=True)

        # 4. Consolidar (inclui Setor)
        df_final = consolidar_dados(df_cvm_full, df_precos, mapa_tickers, mapa_tipo)

        # 5. Exportar (inclui resumos)
        exportar_excel(df_final)
    else:
        print("❌ Nenhum dado CVM encontrado.")